In [1]:
import os
os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

# NOW everything else
import numpy as np
import torch
from replay_buffer import ReplayBuffer
from reward_model_QPA_human import RewardModel
from collections import deque
import utils
import hydra
import wandb
import warnings
import get_human_preferences_v2 as get_human_preferences
import time
warnings.filterwarnings("ignore", category=DeprecationWarning)
os.environ["WANDB_SILENT"] = "true"
warnings.filterwarnings("ignore", category=DeprecationWarning, module="pkg_resources")
warnings.filterwarnings("ignore", category=DeprecationWarning, module="hydra")
os.environ["QT_QPA_PLATFORM"] = "xcb"
%matplotlib qt
PID=1

/home/callie/miniconda3/envs/rl_env/lib/python3.10/site-packages/hydra/_internal/config_loader.py:10: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import (


In [2]:

class Workspace(object):
    def __init__(self, cfg):
        self.work_dir = os.getcwd()
        self.cfg = cfg

        utils.set_seed_everywhere(cfg.seed)
        self.device = torch.device(cfg.device)
        pref_fig = get_human_preferences.create_preference_window()

        self.log_success = False
        # make env
        if 'metaworld' in cfg.env_suite:
            self.env = utils.make_metaworld_env(cfg)
            self.log_success = True
            # need to set seed here
            self.env.action_space.seed(cfg.seed)
            self.env.observation_space.seed(cfg.seed)
            self.env.unwrapped.goal_space.seed(cfg.seed)
        else:
            self.env = utils.make_env(cfg)
        
        self.max_episode_len=self.env.spec.max_episode_steps

        cfg.agent.params.obs_dim = self.env.observation_space.shape[0]
        cfg.agent.params.action_dim = self.env.action_space.shape[0]
        cfg.agent.params.action_range = [
            float(self.env.action_space.low.min()),
            float(self.env.action_space.high.max())
        ]
        self.agent = hydra.utils.instantiate(cfg.agent)

        self.replay_buffer = ReplayBuffer(
            self.env.observation_space.shape,
            self.env.action_space.shape,
            int(cfg.replay_buffer_capacity),
            self.device,
            max_episode_len=self.max_episode_len)
        
        # for logging
        self.total_feedback = 0
        self.labeled_feedback = 0
        self.step = 0

        # instantiating the reward model
        self.reward_model = RewardModel(
            self.env.observation_space.shape[0],
            self.env.action_space.shape[0],
            device=cfg.device,
            ensemble_size=cfg.ensemble_size,
            size_segment=cfg.segment,
            max_size=cfg.max_reward_buffer_size,
            activation=cfg.activation, 
            lr=cfg.reward_lr,
            mb_size=cfg.reward_batch, 
            large_batch=cfg.large_batch, 
            label_margin=cfg.label_margin, 
            teacher_beta=cfg.teacher_beta, 
            teacher_gamma=cfg.teacher_gamma, 
            teacher_eps_mistake=cfg.teacher_eps_mistake, 
            teacher_eps_skip=cfg.teacher_eps_skip, 
            teacher_eps_equal=cfg.teacher_eps_equal,
            data_aug_ratio=cfg.data_aug_ratio,
            max_feedback=cfg.max_feedback,
            pid=PID, 
            use_human_labels=1,
            pref_fig=pref_fig
            )
        
    def evaluate(self):
        average_predicted_episode_reward = 0
        average_true_episode_reward = 0
        success_rate = 0
        
        for episode in range(self.cfg.num_eval_episodes):
            obs, _ = self.env.reset()
            self.agent.reset()
            done = False
            predicted_episode_reward = 0
            true_episode_reward = 0
            if self.log_success:
                episode_success = 0

            while not done:
                with utils.eval_mode(self.agent):
                    action = self.agent.act(obs, sample=False)
                predicted_episode_reward += self.reward_model.r_hat(np.concatenate([obs, action], axis=-1))
                obs, reward, terminated, truncated, extra = self.env.step(action)
                done = terminated or truncated
                true_episode_reward += reward
                if self.log_success:
                    episode_success = max(episode_success, extra['success'])
                
            average_predicted_episode_reward += predicted_episode_reward
            average_true_episode_reward += true_episode_reward
            if self.log_success:
                success_rate += episode_success
            
        average_predicted_episode_reward /= self.cfg.num_eval_episodes
        average_true_episode_reward /= self.cfg.num_eval_episodes
        if self.log_success:
            success_rate /= self.cfg.num_eval_episodes
            success_rate *= 100.0

        return average_true_episode_reward, average_predicted_episode_reward, success_rate

    
    def learn_reward(self, first_flag=0):
                
        # get feedbacks
        labeled_queries, noisy_queries = 0, 0
        # self.reward_model.size_segment += 5
        if first_flag == 1:
            # if it is first time to get feedback, need to use random sampling
            labeled_queries = self.reward_model.uniform_sampling()
        else:
            labeled_queries = self.reward_model.uniform_sampling(self.cfg.explore)
        
        self.total_feedback += self.reward_model.mb_size
        self.labeled_feedback += labeled_queries
        
        train_acc = 0
        if self.labeled_feedback > 0:
            # update reward
            if 'metaworld' in self.cfg.env_suite:
                for _ in range(self.cfg.reward_update):
                    train_acc, reward_loss = self.reward_model.train_reward()
                    total_acc = np.mean(train_acc)
                    if total_acc > 0.97:
                        break;
            else:
                num_iters = int(np.ceil(self.cfg.reward_update*self.labeled_feedback/self.reward_model.train_batch_size))
                train_acc, reward_loss = self.reward_model.train_reward_iter(num_iters)
                total_acc = np.mean(train_acc)
            
            if self.cfg.wandb:
                print('just finished learning reward model')
                wandb.log({'reward_model/train_acc':total_acc, 'reward_model/reward_loss':reward_loss}, step=self.step)


    def run(self):
        episode, train_predicted_episode_reward, done = 0, 0, True
        if self.log_success:
            training_episode_success = 0
        train_true_episode_reward = 0
        train_predicted_episode_reward=0
        
        # store train returns of recent 10 episodes
        avg_train_true_return = deque([], maxlen=10) 

        interact_count = 0
        start_time = time.time()
        #with tqdm(total=self.cfg.num_train_steps) as pbar:
        while self.step < self.cfg.num_train_steps:
            
            if done:
                if self.log_success:
                    wandb.log({'train/true_episode_reward':train_true_episode_reward,
                            'train/predicted_episode_reward':train_predicted_episode_reward,
                            'train/success_rate': training_episode_success,
                            'train/duration': time.time()- start_time}, step=self.step)
                else:
                    wandb.log({'train/true_episode_reward':train_true_episode_reward,
                        'train/predicted_episode_reward':train_predicted_episode_reward,
                        'train/duration': time.time()- start_time}, step=self.step)   
                # evaluate agent periodically
                if self.step > 0 and self.step % self.cfg.eval_frequency == 0:
                    eval_true_episode_reward, eval_predicted_episode_reward, eval_success_rate  = self.evaluate()
                    if self.cfg.wandb:
                        if self.log_success:
                            wandb.log({'eval/true_episode_reward':eval_true_episode_reward,
                                    'eval/predicted_episode_reward':eval_predicted_episode_reward,
                                    'eval/success_rate': eval_success_rate}, step=self.step)

                        else:
                            wandb.log({'eval/true_episode_reward':eval_true_episode_reward,
                                'eval/predicted_episode_reward':eval_predicted_episode_reward}, step=self.step)
                             
                            
                obs, _ = self.env.reset()
                self.agent.reset()
                done = False
                start_time = time.time()
                train_predicted_episode_reward = 0
                avg_train_true_return.append(train_true_episode_reward)
                train_true_episode_reward = 0
                if self.log_success:
                    training_episode_success = 0
                episode_step = 0
                episode += 1
                        
            # sample action for data collection
            if self.step < self.cfg.num_seed_steps:
                action = self.env.action_space.sample()
            else:
                with utils.eval_mode(self.agent):
                    action = self.agent.act(obs, sample=True)

            # run training update                
            if self.step == (self.cfg.num_seed_steps + self.cfg.num_unsup_steps):
                # update schedule
                if self.cfg.reward_schedule == 1:
                    frac = (self.cfg.num_train_steps-self.step) / self.cfg.num_train_steps
                    if frac == 0:
                        frac = 0.01
                elif self.cfg.reward_schedule == 2:
                    frac = self.cfg.num_train_steps / (self.cfg.num_train_steps-self.step +1)
                else:
                    frac = 1
                self.reward_model.change_batch(frac)
                
                # update margin --> not necessary / will be updated soon
                new_margin = np.mean(avg_train_true_return) * (self.cfg.segment / self.max_episode_len)
                self.reward_model.set_teacher_thres_skip(new_margin)
                self.reward_model.set_teacher_thres_equal(new_margin)
                
                # first learn reward
                self.learn_reward(first_flag=1)
                
                # relabel buffer
                self.replay_buffer.relabel_with_predictor(self.reward_model)
                
                # reset Q due to unsuperivsed exploration
                self.agent.reset_critic()
                
                # update agent
                self.agent.update_after_reset(
                    self.replay_buffer, self.step, 
                    gradient_update=self.cfg.reset_update, 
                    policy_update=True)
                
                # reset interact_count
                interact_count = 0
            elif self.step > self.cfg.num_seed_steps + self.cfg.num_unsup_steps:
                # update reward function
                if self.total_feedback < self.cfg.max_feedback:
                    if interact_count == self.cfg.num_interact:
                        # update schedule
                        if self.cfg.reward_schedule == 1:
                            frac = (self.cfg.num_train_steps-self.step) / self.cfg.num_train_steps
                            if frac == 0:
                                frac = 0.01
                        elif self.cfg.reward_schedule == 2:
                            frac = self.cfg.num_train_steps / (self.cfg.num_train_steps-self.step +1)
                        else:
                            frac = 1
                        self.reward_model.change_batch(frac)
                        
                        # update margin --> not necessary / will be updated soon
                        new_margin = np.mean(avg_train_true_return) * (self.cfg.segment / self.max_episode_len)
                        self.reward_model.set_teacher_thres_skip(new_margin * self.cfg.teacher_eps_skip)
                        self.reward_model.set_teacher_thres_equal(new_margin * self.cfg.teacher_eps_equal)
                        
                        # corner case: new total feed > max feed
                        if self.reward_model.mb_size + self.total_feedback > self.cfg.max_feedback:
                            self.reward_model.set_batch(self.cfg.max_feedback - self.total_feedback)
                            
                        self.learn_reward()

                        self.replay_buffer.relabel_with_predictor(self.reward_model)
                        interact_count = 0

                if (self.total_feedback < self.cfg.max_feedback):
                    size = min(self.cfg.max_reward_buffer_size, len(self.reward_model.inputs)-1)
                    self.agent.update_onpolicy_sample(self.replay_buffer, self.step, size, 1, self.cfg.her_ratio)
                else:
                    self.agent.update(self.replay_buffer, self.step, 1)

            # unsupervised exploration
            elif self.step > self.cfg.num_seed_steps:
                self.agent.update_state_ent(self.replay_buffer, self.step, 
                                            gradient_update=1, K=self.cfg.topK)
                
            next_obs, reward, terminated, truncated, extra = self.env.step(action)

            done = terminated or truncated

            if self.total_feedback < self.cfg.max_feedback:
                frame = self.env.render()


            else:
                frame = None 
            reward_hat = self.reward_model.r_hat(np.concatenate([obs, action], axis=-1))


            train_predicted_episode_reward += reward_hat
            train_true_episode_reward += reward 
            
            if self.log_success:
                training_episode_success = max(training_episode_success, extra['success'])
                

            # adding data to the reward training data
            self.reward_model.add_data(obs, action, reward, done, frame)
            self.replay_buffer.add(
                obs, action, reward_hat, 
                next_obs, done, terminated)
            


            obs = next_obs
            episode_step += 1
            self.step += 1
            interact_count += 1
            
        self.agent.save(self.work_dir, self.step)
        self.reward_model.save(self.work_dir, self.step)

In [3]:
from omegaconf import OmegaConf, DictConfig

# Load both files
base_cfg  = OmegaConf.load('./config/train_QPA.yaml')
agent_cfg = OmegaConf.load('./config/agent/sac.yaml')

# Merge agent config into base (agent_cfg keys sit at the top level alongside base keys,
# so interpolations like ${device} and ${agent.params.obs_dim} can resolve correctly)
cfg = OmegaConf.merge(base_cfg, agent_cfg)

# Remove Hydra metadata key — not needed at runtime
OmegaConf.set_struct(cfg, False)
del cfg['defaults']
del cfg['hydra']

print(cfg.pretty())

activation: tanh
agent:
  class: agent.sac.SACAgent
  name: sac
  params:
    action_dim: ???
    action_range: ???
    actor_betas:
    - 0.9
    - 0.999
    actor_cfg: ${diag_gaussian_actor}
    actor_lr: 0.0001
    actor_update_frequency: 1
    alpha_betas:
    - 0.9
    - 0.999
    alpha_lr: 0.0001
    batch_size: 1024
    critic_betas:
    - 0.9
    - 0.999
    critic_cfg: ${double_q_critic}
    critic_lr: 0.0001
    critic_target_update_frequency: 2
    critic_tau: 0.005
    device: ${device}
    discount: 0.99
    init_temperature: 0.1
    learnable_temperature: true
    obs_dim: ???
    wandb: ${wandb}
data_aug_ratio: 20
device: cuda
diag_gaussian_actor:
  class: agent.actor.DiagGaussianActor
  params:
    action_dim: ${agent.params.action_dim}
    hidden_depth: 2
    hidden_dim: 1024
    log_std_bounds:
    - -5
    - 2
    obs_dim: ${agent.params.obs_dim}
double_q_critic:
  class: agent.critic.DoubleQCritic
  params:
    action_dim: ${agent.params.action_dim}
    hidden_depth

In [4]:
# Cell 4 — Run
workspace = Workspace(cfg)
wandb.init(
    project="PbRL_Human_Preferences_Benchmarking",
    config=utils.flatten_dict(dict(cfg)),
    name=f'human_QPA_{cfg.env}_{cfg.max_feedback}_{cfg.seed}',
    entity="musliman",
    mode='online',
)
workspace.run()

rewatch_a: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
rewatch_b: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
rewatch_both: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
(10, 50, 30) (10, 50, 30) (10, 1) (10,)
Saved feedback session 1 to human_feedback_data/1/session_001.pkl
Iter 0, Ensemble Acc: [0.54], Loss: 0.6582868099212646
Iter 1, Ensemble Acc: [0.675], Loss: 0.5657607018947601
Iter 2, Ensemble Acc: [0.72666667], Loss: 0.5037757555643717
Iter 3, Ensemble Acc: [0.77125], Loss: 0.4471356123685837
Iter 4, Ensemble Acc: [0.807], Loss: 0.40636340379714964
Iter 5, Ensemble Acc: [0.83916667], Loss: 0.37075791756312054
Iter 6, Ensemble Acc: [0.86], Loss: 0.34319264122418
Iter 7, Ensemble Acc: [0.87375], Loss: 0.32051512598991394
Iter 8, Ensemble Acc: [0.88555556], Loss: 0.3009608073367013
Iter 9, Ensemble Acc: [0.8955], Loss: 0.28222593367099763
Iter 10, Ensemble Acc: [0.90272727], Loss: 0.2660759071057493
Iter 11, Ensemble Acc: [0.90916667], Loss: 0.2535293797651927
Iter 12, Ensemble Acc: [0.91538462], Loss: 0.2405355744636975

KeyboardInterrupt: 